# Chương 2: Thực nghiệm & Đánh giá thuật toán `negFIN` (Julia)

Notebook này tự động hóa toàn bộ quy trình thực nghiệm Chương 2 trên 5 bộ dữ liệu benchmark chuẩn (**Chess**, **Mushroom**, **Accidents**, **Retail**, **T10I4D100K**) và dữ liệu tổng hợp:
1. **Correctness:** Đối chiếu 100% giữa `negfin_baseline`, `negfin_optimized` và `spmf.jar`.
2. **Runtime theo `minsup`:** 6 mốc `minsup` từ cao xuống thấp cho từng CSDL.
3. **Output Size:** So sánh hiện tượng bùng nổ tổ hợp giữa dữ liệu Dense và Sparse.
4. **Memory (Peak RSS & Heap Allocated):** So sánh trực tiếp Baseline vs. Optimized.
5. **Scalability:** Lát cắt 10%, 25%, 50%, 75%, 100% giao dịch trên `Retail` và `Accidents`.
6. **Ảnh hưởng độ dài giao dịch trung bình:** CSDL tổng hợp (`seed = 42`), liên hệ với kích thước cây BMC-tree và `NegNodeset`.

> **Hướng dẫn cấu hình Hugging Face cho Colab:** Điền tên public Dataset repo của nhóm vào biến `HF_DATASET_REPO` ở cell bên dưới (ví dụ: `"Thanhtung46/fim-benchmark-spmf"`).

In [ ]:
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

# ==============================================================================
# CẤU HÌNH NGUỒN DỮ LIỆU (HUGGING FACE DATASET REPO CỦA NHÓM)
# ==============================================================================
HF_DATASET_REPO = ""  # <-- Điền '<username>/<repo-name>' khi đã upload lên Hugging Face

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BENCH_DIR = ROOT_DIR / "data" / "benchmark"
LOGS_DIR = ROOT_DIR / "results" / "logs"
FIG_DIR = ROOT_DIR / "results" / "figures"
BENCH_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SPMF_FALLBACK_URLS = {
    "chess.txt": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/chess.txt",
    "mushroom.txt": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/mushrooms.txt",
    "accidents.txt": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/accidents.txt",
    "retail.txt": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/retail.txt",
    "T10I4D100K.txt": "https://www.philippe-fournier-viger.com/spmf/publicdatasets/T10I4D100K.txt",
}

# 1. Kiểm tra & cài đặt Julia nếu đang chạy trên Google Colab
if shutil.which("julia") is None:
    print("Đang cài đặt Julia 1.11 LTS cho môi trường Colab...")
    subprocess.run(
        "curl -fsSL https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.3-linux-x86_64.tar.gz "
        "| tar -xz -C /usr/local --strip-components=1",
        shell=True,
        check=True,
    )
print("Julia:", subprocess.check_output(["julia", "--version"], text=True).strip())

# 2. Tự động tải 5 bộ dữ liệu benchmark (ưu tiên Hugging Face, fallback về SPMF)
for fname, fallback_url in SPMF_FALLBACK_URLS.items():
    target = BENCH_DIR / fname
    if target.exists() and target.stat().st_size > 1000:
        print(f"[OK] Đã có sẵn {fname} ({target.stat().st_size / 1e6:.2f} MB)")
        continue
    url = (
        f"https://huggingface.co/datasets/{HF_DATASET_REPO}/resolve/main/{fname}"
        if HF_DATASET_REPO
        else fallback_url
    )
    print(f"Đang tải {fname} từ {url} ...")
    urllib.request.urlretrieve(url, target)
    print(f"  -> Xong ({target.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# Chạy toàn bộ 6 thực nghiệm bằng Julia và xuất kết quả CSV vào results/logs/
cmd = ["julia", f"--project={ROOT_DIR}", str(ROOT_DIR / "scripts" / "run_benchmarks.jl")]
print("Chạy lệnh:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# Thực nghiệm 1: Bảng kiểm tra tính đúng đắn (Correctness)
df_corr = pd.read_csv(LOGS_DIR / "exp1_correctness.csv")
print("=== THỰC NGHIỆM 1: CORRECTNESS ===")
display(df_corr)

In [ ]:
# Thực nghiệm 2: Biểu đồ thời gian chạy theo minsup (Baseline vs. Optimized vs. SPMF)
df_rt = pd.read_csv(LOGS_DIR / "exp2_3_runtime_output.csv")
datasets = df_rt["dataset"].unique()

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
axes = axes.flatten()

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    sub = df_rt[df_rt["dataset"] == ds].sort_values("minsup", ascending=False)
    x = sub["minsup"] * 100.0
    ax.plot(x, sub["time_base_s"], marker="o", linewidth=2, label="negFIN Baseline (Julia)", color="#d95f02")
    ax.plot(x, sub["time_opt_s"], marker="s", linewidth=2, label="negFIN Optimized (Julia)", color="#1b9e77")
    if sub["time_spmf_s"].notna().any():
        ax.plot(x, sub["time_spmf_s"], marker="^", linestyle="--", linewidth=1.8, label="SPMF negFIN (Java)", color="#7570b3")
    ax.invert_xaxis()  # minsup giảm dần từ trái sang phải
    ax.set_title(f"{ds} ({sub['kind'].iloc[0].capitalize()})", fontweight="bold")
    ax.set_xlabel("Minimum Support (%)")
    ax.set_ylabel("Runtime (s)")
    ax.legend(fontsize=8.5)

axes[-1].axis("off")
fig.suptitle("Thực nghiệm 2: Thời gian chạy theo Minimum Support (Baseline vs. Optimized)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig1_runtime_vs_minsup.png", bbox_inches="tight")
plt.show()

In [ ]:
# Thực nghiệm 3: Số lượng tập phổ biến (Output Size) theo minsup — Dense vs. Sparse
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ds in df_rt[df_rt["kind"] == "dense"]["dataset"].unique():
    sub = df_rt[df_rt["dataset"] == ds].sort_values("minsup", ascending=False)
    ax1.plot(sub["minsup"] * 100, sub["n_itemsets"], marker="o", linewidth=2, label=ds)
ax1.invert_xaxis()
ax1.set_yscale("log")
ax1.set_title("Dữ liệu Dày đặc (Dense)", fontweight="bold")
ax1.set_xlabel("Minimum Support (%)")
ax1.set_ylabel("Số lượng Frequent Itemsets (thang log)")
ax1.legend()

for ds in df_rt[df_rt["kind"] == "sparse"]["dataset"].unique():
    sub = df_rt[df_rt["dataset"] == ds].sort_values("minsup", ascending=False)
    ax2.plot(sub["minsup"] * 100, sub["n_itemsets"], marker="s", linewidth=2, label=ds)
ax2.invert_xaxis()
ax2.set_yscale("log")
ax2.set_title("Dữ liệu Thưa (Sparse)", fontweight="bold")
ax2.set_xlabel("Minimum Support (%)")
ax2.set_ylabel("Số lượng Frequent Itemsets (thang log)")
ax2.legend()

fig.suptitle("Thực nghiệm 3: Số lượng tập phổ biến theo Minimum Support (Dense vs. Sparse)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig2_output_size.png", bbox_inches="tight")
plt.show()

In [ ]:
# Thực nghiệm 4: Mức sử dụng bộ nhớ (Heap Allocated & Peak RSS) — Baseline vs. Optimized
df_mem = pd.read_csv(LOGS_DIR / "exp4_memory.csv")
display(df_mem)

x = np.arange(len(df_mem))
width = 0.36
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5))

ax1.bar(x - width / 2, df_mem["alloc_base_mb"], width, label="Baseline", color="#d95f02")
ax1.bar(x + width / 2, df_mem["alloc_opt_mb"], width, label="Optimized", color="#1b9e77")
ax1.set_xticks(x)
ax1.set_xticklabels(df_mem["dataset"], rotation=15)
ax1.set_ylabel("Tổng Heap Cấp Phát (MB)")
ax1.set_title("Heap Allocation (@allocated)", fontweight="bold")
ax1.legend()

ax2.bar(x - width / 2, df_mem["peak_rss_base_mb"], width, label="Baseline", color="#d95f02")
ax2.bar(x + width / 2, df_mem["peak_rss_opt_mb"], width, label="Optimized", color="#1b9e77")
ax2.set_xticks(x)
ax2.set_xticklabels(df_mem["dataset"], rotation=15)
ax2.set_ylabel("Peak RSS (MB)")
ax2.set_title("RAM Đỉnh Tiến Trình (Peak RSS)", fontweight="bold")
ax2.legend()

fig.suptitle("Thực nghiệm 4: So sánh Bộ nhớ giữa Baseline và Optimized", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig3_memory_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Thực nghiệm 5: Khả năng mở rộng (Scalability 10% -> 100% trên Retail & Accidents)
df_scale = pd.read_csv(LOGS_DIR / "exp5_scalability.csv")
display(df_scale)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ds in zip(axes, df_scale["dataset"].unique()):
    sub = df_scale[df_scale["dataset"] == ds]
    pct = sub["fraction"] * 100
    ax.plot(pct, sub["time_base_s"], marker="o", linewidth=2, label="Baseline", color="#d95f02")
    ax.plot(pct, sub["time_opt_s"], marker="s", linewidth=2, label="Optimized", color="#1b9e77")
    ax.set_title(f"Scalability — {ds} (minsup={sub['minsup'].iloc[0]*100:.1f}%)", fontweight="bold")
    ax.set_xlabel("Tỷ lệ kích thước CSDL (%)")
    ax.set_ylabel("Runtime (s)")
    ax.set_xticks([10, 25, 50, 75, 100])
    ax.legend()

fig.suptitle("Thực nghiệm 5: Khả năng mở rộng theo số lượng giao dịch", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig4_scalability.png", bbox_inches="tight")
plt.show()

In [ ]:
# Thực nghiệm 6: Ảnh hưởng độ dài giao dịch trung bình (Synthetic, seed=42)
df_len = pd.read_csv(LOGS_DIR / "exp6_tx_length.csv")
display(df_len)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5))

ax1.plot(df_len["avg_len"], df_len["time_base_s"], marker="o", linewidth=2, label="Baseline", color="#d95f02")
ax1.plot(df_len["avg_len"], df_len["time_opt_s"], marker="s", linewidth=2, label="Optimized", color="#1b9e77")
ax1.set_title("Thời gian chạy theo Độ dài giao dịch TB", fontweight="bold")
ax1.set_xlabel("Độ dài giao dịch trung bình (#items/tx)")
ax1.set_ylabel("Runtime (s)")
ax1.legend()

color_nodes = "#386cb0"
color_is = "#7fc97f"
ax2.plot(df_len["avg_len"], df_len["bmc_nodes"], marker="^", linewidth=2, color=color_nodes, label="Số nút BMC-tree")
ax2.set_xlabel("Độ dài giao dịch trung bình (#items/tx)")
ax2.set_ylabel("Số nút BMC-tree", color=color_nodes)
ax2.tick_params(axis="y", labelcolor=color_nodes)
ax2_r = ax2.twinx()
ax2_r.plot(df_len["avg_len"], df_len["n_itemsets"], marker="D", linestyle="--", linewidth=2, color="#e7298a", label="Số Frequent Itemsets")
ax2_r.set_ylabel("Số Frequent Itemsets", color="#e7298a")
ax2_r.tick_params(axis="y", labelcolor="#e7298a")
ax2.set_title("Phình to BMC-tree & Bùng nổ Itemsets", fontweight="bold")

fig.suptitle("Thực nghiệm 6: Ảnh hưởng của độ dài giao dịch trung bình (10,000 tx, 80 items)", fontsize=13, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig5_tx_length.png", bbox_inches="tight")
plt.show()

## Phân tích chuyên sâu & Đề xuất hướng tối ưu tiếp theo

1. **Giải thích dựa trên lý thuyết `negFIN` (Aryabarzan et al., 2018):**
   - Độ phức tạp khai thác của `negFIN` là $O(l \cdot n)$ (thay vì $O(l(m+n))$ của dFIN) nhờ mỗi `NegNodeset` được trích từ **duy nhất 1 tập nút** bằng phép toán bitwise trong $O(1)$.
   - Bản **Optimized** vượt trội rõ rệt so với **Baseline** nhờ: (i) chuyên biệt hóa bit-packing (`UInt64`/`UInt128`/`NTuple`/`Vector{UInt64}`) qua **Multiple Dispatch**, (ii) cây BMC-tree dạng LCRS giảm mạnh cấp phát cho nút lá trên dữ liệu thưa (`Retail`, `T10I4D100K`), (iii) kỹ thuật **Early-Abort** dừng ngay khi `neg_sup > parent_sup - min_sup_count`, và (iv) cơ chế xuất **Promotion** không cấp phát `FIS_parent`.
   - Khi tăng **độ dài giao dịch trung bình** (Thực nghiệm 6), khả năng chia sẻ tiền tố trên BMC-tree giảm mạnh khiến số nút BMC-tree và kích thước `NegNodeset` $n$ tăng đồng thời, làm tăng nhanh cả thời gian lẫn bộ nhớ.

2. **Đề xuất $\ge 2$ hướng tối ưu hóa tiếp theo:**
   - **Đề xuất 1 — Song song hóa đa luồng (`Threads.@threads`) ở tầng 1 của Set-Enumeration Tree:** Các cây con xuất phát từ từng nút ở tầng 1 độc lập hoàn toàn về trạng thái ghi (chỉ đọc `Nodeset`), có thể phân phối động (dynamic scheduling) để cân bằng tải giữa các nhánh nặng và nhẹ.
   - **Đề xuất 2 — Nén Bitmap theo đoạn (Trimmed / Sparse Bitset) cho dữ liệu thưa có $n_f$ lớn:** Theo Property 1, phần *don't-care section* luôn bằng 0; chỉ lưu dải `UInt64` word khác 0 (`offset` + `words`) giúp giảm mạnh dung lượng RAM trên `Retail` hoặc `Kosarak` ở ngưỡng `minsup` rất thấp.